In [15]:
import pandas as pd

In [16]:
df=pd.read_csv('Symptoms2Disease\Symptom2Disease.csv')

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\arbaz\AppData\Local\Temp\ipykernel_17384\2440158164.py:1: SyntaxWarning: invalid escape sequence '\S'
  df=pd.read_csv('Symptoms2Disease\Symptom2Disease.csv')


In [3]:
df

,Unnamed: 0,label,text
0,0,Psoriasis,I have been experiencing a skin rash on my arm...
1,1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,2,Psoriasis,I have been experiencing joint pain in my fing...
3,3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,4,Psoriasis,"My nails have small dents or pits in them, and..."
...,...,...,...
1195,295,diabetes,I'm shaking and trembling all over. I've lost ...
1196,296,diabetes,"Particularly in the crevices of my skin, I hav..."
1197,297,diabetes,I regularly experience these intense urges and...
1198,298,diabetes,"I have trouble breathing, especially outside. ..."


In [8]:
df['label'].unique()

<ArrowStringArray>
[                      'Psoriasis',                  'Varicose Veins',
                         'Typhoid',                     'Chicken pox',
                        'Impetigo',                          'Dengue',
                'Fungal infection',                     'Common Cold',
                       'Pneumonia',           'Dimorphic Hemorrhoids',
                       'Arthritis',                            'Acne',
                'Bronchial Asthma',                    'Hypertension',
                        'Migraine',            'Cervical spondylosis',
                        'Jaundice',                         'Malaria',
         'urinary tract infection',                         'allergy',
 'gastroesophageal reflux disease',                   'drug reaction',
            'peptic ulcer disease',                        'diabetes']
Length: 24, dtype: str

In [12]:
import pandas as pd

df = pd.read_csv("emergency_dataset.csv")

print("Shape:", df.shape)
print(df.head())
print(df["label"].value_counts())

Shape: (0, 2)
Empty DataFrame
Columns: [text, label]
Index: []
Series([], Name: count, dtype: int64)


In [17]:
print(df["label"].unique())


<ArrowStringArray>
[                      'Psoriasis',                  'Varicose Veins',
                         'Typhoid',                     'Chicken pox',
                        'Impetigo',                          'Dengue',
                'Fungal infection',                     'Common Cold',
                       'Pneumonia',           'Dimorphic Hemorrhoids',
                       'Arthritis',                            'Acne',
                'Bronchial Asthma',                    'Hypertension',
                        'Migraine',            'Cervical spondylosis',
                        'Jaundice',                         'Malaria',
         'urinary tract infection',                         'allergy',
 'gastroesophageal reflux disease',                   'drug reaction',
            'peptic ulcer disease',                        'diabetes']
Length: 24, dtype: str


In [18]:
mapping = {
    # HIGH RISK (2)
    "pneumonia": 2,
    "dengue": 2,
    "typhoid": 2,
    "bronchial asthma": 2,
    "malaria": 2,
    "chicken pox": 2,
    "drug reaction": 2,
    "urinary tract infection": 2,
    "peptic ulcer disease": 2,

    # MEDIUM RISK (1)
    "diabetes": 1,
    "hypertension": 1,
    "migraine": 1,
    "jaundice": 1,
    "arthritis": 1,
    "common cold": 1,
    "cervical spondylosis": 1,
    "gastroesophageal reflux disease": 1,
    "dimorphic hemorrhoids": 1,

    # LOW RISK (0)
    "acne": 0,
    "fungal infection": 0,
    "allergy": 0,
    "impetigo": 0,
    "psoriasis": 0,
    "varicose veins": 0
}

In [19]:
df["label"] = df["label"].astype(str).str.strip().str.lower()

In [20]:
# clean labels
df["label"] = df["label"].astype(str).str.strip().str.lower()

# map
df["label"] = df["label"].map(mapping)

# debug
print(df["label"].value_counts(dropna=False))

# remove missing
df = df.dropna(subset=["label"])

# safety check
if len(df) == 0:
    raise ValueError("Still empty! mapping mismatch")

# convert
df["label"] = df["label"].astype(int)

# final dataset
df = df[["text", "label"]]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("DONE:", len(df))

label
2    450
1    450
0    300
Name: count, dtype: int64
DONE: 1200


In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch

# Load dataset


# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function
def tokenize(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# Dataset class
class EmergencyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmergencyDataset(train_encodings, train_labels)
val_dataset = EmergencyDataset(val_encodings, val_labels)

# Load model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

# Training settings
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train model
trainer.train()

# Save model
model.save_pretrained("emergency_model")
tokenizer.save_pretrained("emergency_model")

d:\AI emergency trigger\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5571.23it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Step,Training Loss
10,1.086000
20,1.090517
30,1.082000
40,1.040209
50,0.894088
60,0.589172
70,0.619927
80,0.670855
90,0.506608
100,0.380977


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]


('emergency_model\\tokenizer_config.json', 'emergency_model\\tokenizer.json')

In [22]:
from transformers import pipeline
import torch

# 1. check device (CPU or GPU)
device = 0 if torch.cuda.is_available() else -1

# 2. load model safely
model = pipeline(
    task="text-classification",
    model="emergency_model",
    tokenizer="emergency_model",
    device=device,
    return_all_scores=True   # shows probability of all classes
)

# 3. test inputs
texts = [
    "Patient not breathing",
    "Fever for 2 days",
    "Need hospital address",
    "Severe chest pain and unconscious",
    "Mild headache only"
]

# 4. run predictions
for text in texts:
    result = model(text)
    print("\nText:", text)
    print("Prediction:", result)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6117.66it/s]



Text: Patient not breathing
Prediction: [{'label': 'LABEL_1', 'score': 0.8698078393936157}]

Text: Fever for 2 days
Prediction: [{'label': 'LABEL_2', 'score': 0.993999719619751}]

Text: Need hospital address
Prediction: [{'label': 'LABEL_2', 'score': 0.5512176156044006}]

Text: Severe chest pain and unconscious
Prediction: [{'label': 'LABEL_1', 'score': 0.9403611421585083}]

Text: Mild headache only
Prediction: [{'label': 'LABEL_2', 'score': 0.9913564920425415}]


In [23]:
id2label = {
    0: "LOW RISK",
    1: "MEDIUM RISK",
    2: "HIGH RISK"
}

label2id = {
    "LOW RISK": 0,
    "MEDIUM RISK": 1,
    "HIGH RISK": 2
}

In [24]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

model = pipeline(
    "text-classification",
    model="emergency_model",
    tokenizer="emergency_model",
    device=device,
    return_all_scores=True
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4080.02it/s]


In [25]:
texts = [
    "Patient not breathing",
    "Fever for 2 days",
    "Need hospital address",
    "Severe chest pain and unconscious",
    "Mild headache only"
]

for text in texts:
    result = model(text)[0]

    best = max(result, key=lambda x: x["score"])

    label_id = int(best["label"].split("_")[-1])

    print("\nText:", text)
    print("Prediction:", id2label[label_id])
    print("Confidence:", round(best["score"], 4))

TypeError: string indices must be integers, not 'str'

In [27]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

model = pipeline(
    "text-classification",
    model="emergency_model",
    tokenizer="emergency_model",
    device=device,
    return_all_scores=True
)

id2label = {
    0: "LOW RISK",
    1: "MEDIUM RISK",
    2: "HIGH RISK"
}

texts = [
    "Patient not breathing",
    "Severe chest pain",
    "Fever for 2 days",
    "Mild headache",
    "Need hospital address"
]

for text in texts:
    scores = model(text)[0]
    best = max(scores, key=lambda x: x["score"])

    label_id = int(best["label"].split("_")[-1])

    print(f"\nText: {text}")
    print(f"Prediction: {id2label[label_id]}")
    print(f"Confidence: {best['score']:.4f}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6500.47it/s]


TypeError: string indices must be integers, not 'str'